# Architectural Electrical Plan Sheet Detection with YOLO
### End-to-End Pipeline for Vector PDF Rasterization, High-Resolution Slicing (SAHI), Training, and Structured Takeoff Extraction

This notebook demonstrates how to process large architectural plan drawings (PDF / CAD exports), detect small electrical symbols (switches, receptacles, lights, life safety devices), deduplicate detections across overlapping window seams, and extract structured JSON data.

In [ ]:
import sys
import os
import json
from pathlib import Path
import cv2
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.pdf_processor import PlanSheetPreprocessor
from src.tiler import PlanSheetTiler
from src.nms_stitcher import CoordinateStitcher
from src.visualizer import PlanSheetVisualizer
from src.inference import ArchitecturalPlanDetector

print("All modules loaded successfully!")

## 1. High-Resolution PDF Rasterization
Architectural drawings are typically large format ($24'' \times 36''$ or $36'' \times 48''$). We rasterize vector PDFs at 300 DPI to preserve crisp line work and tiny electrical glyphs.

In [ ]:
sample_path = PROJECT_ROOT / "data" / "raw" / "sample_plan.jpg"
preprocessor = PlanSheetPreprocessor(target_dpi=300)
sheet_img, meta = preprocessor.load_or_rasterize(sample_path)

print(f"Loaded Drawing: {meta['file_name']}")
print(f"Resolution: {meta['width']} x {meta['height']} pixels")

# Display preview
plt.figure(figsize=(10, 7))
plt.imshow(cv2.cvtColor(sheet_img, cv2.COLOR_BGR2RGB))
plt.title(f"{meta['file_name']} - Original Plan Sheet")
plt.axis("off")
plt.show()

## 2. Sliding Window Tiling (SAHI Pattern)
To prevent small symbols from being downsampled into unreadable noise, we generate $1024 \times 1024$ (or $512 \times 512$) tiles with a $25\%$ overlap across horizontal and vertical dimensions.

In [ ]:
tiler = PlanSheetTiler(slice_size=512, overlap_ratio=0.25)
h, w = sheet_img.shape[:2]
windows = tiler.compute_slice_windows(h, w)
print(f"Total sliding-window tiles generated: {len(windows)}")
for i, win in enumerate(windows):
    print(f"  Tile {i}: X=[{win[0]}, {win[2]}], Y=[{win[1]}, {win[3]}]")

## 3. YOLO Model Training
Fine-tuning YOLO (YOLOv8 / YOLO11) with CAD drafting-specific hyperparameters (line dropout, 360-degree rotation invariance, affine scale jitter, and reduced mosaic to protect circuit arcs).

In [ ]:
from ultralytics import YOLO

# Initialize YOLO model
model = YOLO("yolo11n.pt")

# Sample training invocation:
# results = model.train(
#     data=str(PROJECT_ROOT / "configs" / "dataset.yaml"),
#     cfg=str(PROJECT_ROOT / "configs" / "hyp_cad.yaml"),
#     epochs=100,
#     imgsz=1024,
#     batch=8,
#     project=str(PROJECT_ROOT / "runs" / "train"),
#     name="electrical_cad_yolo"
# )
print("Model initialized and ready for training.")

## 4. Structured JSON Output & Takeoff Summary
Detections are deduplicated across seams using global Non-Maximum Suppression and exported into the project specification JSON format.

In [ ]:
json_path = PROJECT_ROOT / "output" / "sample_detection.json"
with open(json_path, "r", encoding="utf-8") as f:
    takeoff_data = json.load(f)

print("Bill of Materials Takeoff:")
for symbol, count in takeoff_data["summary_counts"].items():
    print(f"  - {symbol:<25}: {count}")

print(f"\nTotal detected symbols: {takeoff_data['total_count']}")

## 5. Visual Takeoff Overlay
Bounding boxes, classification tags, and confidence scores rendered back onto the original high-resolution sheet for engineering quality assurance.

In [ ]:
annotated_path = PROJECT_ROOT / "output" / "sample_annotated_takeoff.png"
if annotated_path.exists():
    annotated_img = cv2.imread(str(annotated_path))
    plt.figure(figsize=(14, 10))
    plt.imshow(cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB))
    plt.title("Visual Verification Overlay with Detected Electrical Symbols")
    plt.axis("off")
    plt.show()
else:
    print("Run 'python main.py demo' to generate the annotated verification image.")